In [ ]:
# PROJETO DE ZOONOSES — GERADOR DE DADOS BRUTOS
# Versão revisada: geração incremental, problemas controlados,
# duplicidades e relacionamentos por identificador técnico.

import os
import re
import json
import random
import sqlite3
import math
from copy import deepcopy
from datetime import date, datetime, timedelta
from collections import Counter

# 1. CONFIGURAÇÕES

PASTA_PROJETO = "/content/PROJETO_ZOONOSES"
CAMINHO_BANCO = os.path.join(
    PASTA_PROJETO,
    "banco_zoonoses_v2.db"
)

# Defina aqui quantos animais deseja gerar neste lote.
QUANTIDADE_ANIMAIS = 5000

TAXA_PROBLEMAS = 0.12       # 12% dos registros originais de cada conjunto
TAXA_DUPLICADOS = 0.02      # 2% adicionais de animais duplicados

MODO_GERACAO = "INICIAL"
# INICIAL: usar somente na primeira geracao deste banco novo.
# ADICIONAR: usar nas proximas geracoes, acrescentando outro lote
# sem apagar os registros anteriores.

random.seed()


# 2. CONEXAO COM O BANCO LOCAL DO COLAB — SEM GOOGLE DRIVE

if not os.path.isdir(PASTA_PROJETO):
    raise FileNotFoundError(
        "Pasta do projeto nao encontrada. Execute primeiro "
        "o notebook BANCO_DE_DADOS_ZOONOSES."
    )

if not os.path.exists(CAMINHO_BANCO):
    raise FileNotFoundError(
        "Banco SQLite não encontrado. Confira o caminho do arquivo."
    )

if not isinstance(QUANTIDADE_ANIMAIS, int) or QUANTIDADE_ANIMAIS <= 0:
    raise ValueError(
        "QUANTIDADE_ANIMAIS precisa ser um numero inteiro maior que zero."
    )

if MODO_GERACAO not in {"INICIAL", "ADICIONAR"}:
    raise ValueError("MODO_GERACAO deve ser 'INICIAL' ou 'ADICIONAR'.")

conexao = sqlite3.connect(CAMINHO_BANCO, timeout=60)
conexao.execute("PRAGMA foreign_keys = ON")
conexao.execute("PRAGMA busy_timeout = 60000")
cursor = conexao.cursor()


# 3. VERIFICAÇÃO DAS TABELAS

tabelas_necessarias = [
    "lotes",
    "registros_brutos",
    "auditoria"
]

tabelas_existentes = {
    linha[0]
    for linha in cursor.execute(
        "SELECT name FROM sqlite_master WHERE type='table'"
    ).fetchall()
}

for tabela in tabelas_necessarias:
    if tabela not in tabelas_existentes:
        conexao.close()
        raise Exception(
            f"A tabela '{tabela}' não foi encontrada no banco."
        )

# Tabela auxiliar para manter a sequência de IDs entre os lotes.
# CREATE IF NOT EXISTS não apaga nenhuma estrutura existente.

cursor.execute("""
CREATE TABLE IF NOT EXISTS controle_sequencias (
    nome TEXT PRIMARY KEY,
    proximo_valor INTEGER NOT NULL
)
""")

cursor.execute("""
INSERT OR IGNORE INTO controle_sequencias
(nome, proximo_valor)
VALUES ('animais', 1)
""")

conexao.commit()


# 4. REFERÊNCIAS PARA OS DADOS SINTÉTICOS

municipios = [
    "Feijó",
    "Cruzeiro do Sul",
    "Sena Madureira",
    "Rio Branco",
    "Tarauacá"
]

especies = ["Cachorro", "Gato"]

nomes_cachorros = [
    "Rex", "Thor", "Bob", "Max", "Toby", "Bidu",
    "Pingo", "Fred", "Zeus", "Billy", "Scooby",
    "Bolinha", "Tito", "Apolo", "Bento", "Nino",
    "Marley", "Simba", "Lucky", "Pipoca",
    "Caramelo", "Duque", "Costelinha", "Toddy",
    "Paçoca", "Pudim", "Fubá", "Boris"
]

nomes_gatos = [
    "Mimi", "Luna", "Nina", "Mel", "Mingau",
    "Frajola", "Tom", "Garfield", "Chico",
    "Pérola", "Amora", "Salem", "Fumaça",
    "Pipoca", "Biscoito", "Tigrinho", "Nala",
    "Simba", "Mittens", "Félix", "Pudim",
    "Bolinha", "Manchinha", "Cacau", "Floquinho"
]

racas_cachorros = [
    "Sem raça definida",
    "Pinscher",
    "Poodle",
    "Labrador",
    "Golden Retriever",
    "Pastor Alemão",
    "Shih-tzu",
    "Dachshund",
    "Pit Bull",
    "Beagle",
    "Border Collie",
    "Rottweiler"
]

racas_gatos = [
    "Sem raça definida",
    "Siamês",
    "Persa",
    "Maine Coon",
    "Angorá",
    "Bengal",
    "Ragdoll"
]

motivos_acolhimento = [
    "Maus-tratos",
    "Abandono",
    "Acidente",
    "Doença",
    "Recolhimento preventivo",
    "Entrega voluntária",
    "Animal perdido",
    "Resgate de filhotes",
    "Outros"
]

origens_acolhimento = [
    "Resgate pela equipe",
    "Denúncia da população",
    "Entrega pelo tutor",
    "Órgão público",
    "Clínica veterinária",
    "Recolhimento nas ruas"
]

vacinas_disponiveis = [
    "Antirrábica",
    "V8",
    "V10",
    "Polivalente felina",
    "Leucemia felina"
]

diagnosticos_disponiveis = [
    "Leishmaniose",
    "Esporotricose",
    "Erliquiose",
    "Cinomose",
    "Parvovirose",
    "Dermatite",
    "Sarna",
    "Verminose",
    "Infecção respiratória",
    "Gastroenterite"
]

tipos_maus_tratos = [
    "Abandono",
    "Agressão física",
    "Negligência",
    "Falta de alimentação",
    "Falta de água",
    "Confinamento inadequado",
    "Ausência de atendimento veterinário"
]

motivos_obito = [
    "Doença",
    "Acidente",
    "Complicações clínicas",
    "Causas naturais"
]

justificativas_eutanasia = [
    "Sofrimento irreversível",
    "Doença em estágio terminal",
    "Condição clínica sem possibilidade de recuperação"
]


# 5. FUNÇÕES AUXILIARES

def data_aleatoria(inicio="2022-01-01", fim=None):
    if fim is None:
        fim = date.today().isoformat()

    inicio_data = datetime.strptime(inicio, "%Y-%m-%d").date()
    fim_data = datetime.strptime(fim, "%Y-%m-%d").date()

    diferenca = (fim_data - inicio_data).days

    return (
        inicio_data + timedelta(days=random.randint(0, diferenca))
    ).isoformat()


def data_futura(dias=90):
    return (
        date.today() + timedelta(days=random.randint(1, dias))
    ).isoformat()


def peso_aleatorio(especie, raca):
    if especie == "Gato":
        return round(random.uniform(2.0, 8.0), 2)

    if raca == "Pinscher":
        return round(random.uniform(2.0, 6.0), 2)

    if raca == "Dachshund":
        return round(random.uniform(4.0, 12.0), 2)

    if raca in ["Labrador", "Golden Retriever", "Pastor Alemão",
                "Rottweiler"]:
        return round(random.uniform(18.0, 38.0), 2)

    if raca == "Pit Bull":
        return round(random.uniform(15.0, 35.0), 2)

    return round(random.uniform(3.0, 25.0), 2)


def criar_meta(id_animal_origem=None):
    return {
        "id_animal_origem": id_animal_origem,
        "problemas_introduzidos": []
    }


def registrar_problema(registro, problema):
    registro.setdefault("_meta", {})
    registro["_meta"].setdefault("problemas_introduzidos", [])
    registro["_meta"]["problemas_introduzidos"].append(problema)


# 6. GERAÇÃO DE UM ANIMAL COERENTE

def gerar_animal(numero):

    especie = random.choice(especies)

    if especie == "Cachorro":
        nome = random.choice(nomes_cachorros)
        raca = random.choice(racas_cachorros)
    else:
        nome = random.choice(nomes_gatos)
        raca = random.choice(racas_gatos)

    municipio = random.choice(municipios)

    data_acolhimento = data_aleatoria()

    animal = {
        "id_animal": f"ANM{numero:08d}",
        "nome": nome,
        "especie": especie,
        "raca": raca,
        "sexo": random.choice(["Macho", "Fêmea"]),
        "municipio": municipio,
        "peso_kg": peso_aleatorio(especie, raca),
        "status": "Ativo",
        "vacinas_faltantes": "",
        "situacao_vacinacao": random.choice([
            "Em dia",
            "Pendente",
            "Parcial"
        ]),
        "imagem": f"{municipio.replace(' ', '_')}_animal_{numero:08d}.jpg",
        "data_acolhimento": data_acolhimento,
        "motivo_acolhimento": random.choice(motivos_acolhimento),
        "origem_acolhimento": random.choice(origens_acolhimento),
        "_meta": criar_meta(f"ANM{numero:08d}")
    }

    return animal


# 7. GERAÇÃO DAS VACINAÇÕES

def gerar_vacinacoes(animal):

    registros = []

    quantidade = random.choices(
        [0, 1, 2, 3],
        weights=[15, 30, 35, 20]
    )[0]

    data_acolhimento = animal["data_acolhimento"]

    for _ in range(quantidade):

        data_aplicacao = data_aleatoria(
            inicio=data_acolhimento
        )

        proxima_dose = (
            datetime.strptime(data_aplicacao, "%Y-%m-%d").date()
            + timedelta(days=random.choice([30, 60, 90, 365]))
        ).isoformat()

        registros.append({
            "id_animal": animal["id_animal"],
            "nome_vacina": random.choice(vacinas_disponiveis),
            "data_aplicacao": data_aplicacao,
            "proxima_dose": proxima_dose,
            "situacao": random.choice([
                "Aplicada",
                "Pendente",
                "Aguardando reforço"
            ]),
            "observacoes": "Registro sintético de vacinação.",
            "_meta": criar_meta(
                animal["_meta"]["id_animal_origem"]
            )
        })

    return registros


# 8. GERAÇÃO DOS DIAGNÓSTICOS

def gerar_diagnosticos(animal):

    registros = []

    # Aproximadamente 12% dos animais recebem diagnóstico.
    if random.random() < 0.12:

        registros.append({
            "id_animal": animal["id_animal"],
            "data_diagnostico": data_aleatoria(
                inicio=animal["data_acolhimento"]
            ),
            "diagnostico": random.choice(
                diagnosticos_disponiveis
            ),
            "descricao": "Diagnóstico sintético para fins acadêmicos.",
            "situacao": random.choice([
                "Em tratamento",
                "Em acompanhamento",
                "Recuperado"
            ]),
            "origem": "Gerado artificialmente",
            "_meta": criar_meta(
                animal["_meta"]["id_animal_origem"]
            )
        })

    return registros


# 9. GERAÇÃO DAS OCORRÊNCIAS DE MAUS-TRATOS

def gerar_maus_tratos(animal):

    registros = []

    motivo = animal["motivo_acolhimento"]

    gerar_ocorrencia = (
        motivo == "Maus-tratos"
        or random.random() < 0.025
    )

    if gerar_ocorrencia:

        registros.append({
            "id_animal": animal["id_animal"],
            "municipio": animal["municipio"],
            "data_ocorrencia": data_aleatoria(
                inicio=animal["data_acolhimento"]
            ),
            "tipo_ocorrencia": random.choice(tipos_maus_tratos),
            "descricao": "Ocorrência sintética para análise.",
            "situacao": random.choice([
                "Em investigação",
                "Em acompanhamento",
                "Concluída"
            ]),
            "origem": "Gerado artificialmente",
            "_meta": criar_meta(
                animal["_meta"]["id_animal_origem"]
            )
        })

    return registros


# 10. GERAÇÃO DE ÓBITOS E EUTANÁSIA

def gerar_eventos(animal):

    registros = []

    # Taxa baixa de eventos, para manter uma base plausível.
    if random.random() < 0.015:

        tipo = random.choices(
            ["Óbito", "Eutanásia"],
            weights=[85, 15]
        )[0]

        if tipo == "Óbito":
            motivo = random.choice(motivos_obito)
            observacoes = "Evento sintético de óbito."
        else:
            motivo = random.choice(justificativas_eutanasia)
            observacoes = "Evento sintético de eutanásia."

        data_evento = data_aleatoria(
            inicio=animal["data_acolhimento"]
        )

        animal["status"] = "Inativo"

        registros.append({
            "id_animal": animal["id_animal"],
            "tipo_evento": tipo,
            "data_evento": data_evento,
            "municipio": animal["municipio"],
            "motivo": motivo,
            "observacoes": observacoes,
            "_meta": criar_meta(
                animal["_meta"]["id_animal_origem"]
            )
        })

    return registros


# 11. INTRODUÇÃO CONTROLADA DE PROBLEMAS

def aplicar_problemas(registros, tipo, taxa):

    if not registros:
        return 0

    quantidade = math.ceil(len(registros) * taxa)

    selecionados = random.sample(
        range(len(registros)),
        min(quantidade, len(registros))
    )

    for indice in selecionados:

        registro = registros[indice]

        if tipo == "animais":

            problema = random.choice([
                "nome_nulo",
                "raca_vazia",
                "peso_negativo",
                "peso_texto",
                "municipio_invalido",
                "sexo_invalido",
                "status_invalido",
                "situacao_vacinacao_invalida",
                "data_acolhimento_futura",
                "motivo_acolhimento_invalido",
                "origem_acolhimento_invalida",
                "id_animal_ausente",
                "id_animal_malformado"
            ])

            if problema == "nome_nulo":
                registro["nome"] = None

            elif problema == "raca_vazia":
                registro["raca"] = ""

            elif problema == "peso_negativo":
                registro["peso_kg"] = -random.uniform(1, 20)

            elif problema == "peso_texto":
                registro["peso_kg"] = "peso desconhecido"

            elif problema == "municipio_invalido":
                registro["municipio"] = "Município inexistente"

            elif problema == "sexo_invalido":
                registro["sexo"] = "Não identificado??"

            elif problema == "status_invalido":
                registro["status"] = "Desconhecido"

            elif problema == "situacao_vacinacao_invalida":
                registro["situacao_vacinacao"] = "Vacinação incorreta"

            elif problema == "data_acolhimento_futura":
                registro["data_acolhimento"] = data_futura()

            elif problema == "motivo_acolhimento_invalido":
                registro["motivo_acolhimento"] = "Motivo não cadastrado"

            elif problema == "origem_acolhimento_invalida":
                registro["origem_acolhimento"] = "Origem desconhecida"

            elif problema == "id_animal_ausente":
                registro["id_animal"] = None

            elif problema == "id_animal_malformado":
                registro["id_animal"] = "ID-ANIMAL-ERRO"

            registrar_problema(registro, problema)

        elif tipo == "vacinacoes":

            problema = random.choice([
                "vacina_nula",
                "data_aplicacao_futura",
                "proxima_dose_anterior",
                "situacao_invalida",
                "id_animal_ausente"
            ])

            if problema == "vacina_nula":
                registro["nome_vacina"] = None

            elif problema == "data_aplicacao_futura":
                registro["data_aplicacao"] = data_futura()

            elif problema == "proxima_dose_anterior":
                registro["proxima_dose"] = "2020-01-01"

            elif problema == "situacao_invalida":
                registro["situacao"] = "Situação desconhecida"

            elif problema == "id_animal_ausente":
                registro["id_animal"] = None

            registrar_problema(registro, problema)

        elif tipo == "diagnosticos":

            problema = random.choice([
                "diagnostico_vazio",
                "data_diagnostico_futura",
                "situacao_invalida",
                "id_animal_ausente"
            ])

            if problema == "diagnostico_vazio":
                registro["diagnostico"] = ""

            elif problema == "data_diagnostico_futura":
                registro["data_diagnostico"] = data_futura()

            elif problema == "situacao_invalida":
                registro["situacao"] = "Situação incorreta"

            elif problema == "id_animal_ausente":
                registro["id_animal"] = None

            registrar_problema(registro, problema)

        elif tipo == "maus_tratos":

            problema = random.choice([
                "municipio_invalido",
                "tipo_ocorrencia_vazio",
                "data_ocorrencia_futura",
                "situacao_invalida",
                "id_animal_ausente"
            ])

            if problema == "municipio_invalido":
                registro["municipio"] = "Local desconhecido"

            elif problema == "tipo_ocorrencia_vazio":
                registro["tipo_ocorrencia"] = ""

            elif problema == "data_ocorrencia_futura":
                registro["data_ocorrencia"] = data_futura()

            elif problema == "situacao_invalida":
                registro["situacao"] = "Situação não cadastrada"

            elif problema == "id_animal_ausente":
                registro["id_animal"] = None

            registrar_problema(registro, problema)

        elif tipo == "obitos_eutanasias":

            problema = random.choice([
                "motivo_ausente",
                "data_evento_anterior_ao_acolhimento",
                "municipio_invalido",
                "tipo_evento_invalido",
                "id_animal_ausente"
            ])

            if problema == "motivo_ausente":
                registro["motivo"] = None

            elif problema == "data_evento_anterior_ao_acolhimento":
                registro["data_evento"] = "2020-01-01"

            elif problema == "municipio_invalido":
                registro["municipio"] = "Município inexistente"

            elif problema == "tipo_evento_invalido":
                registro["tipo_evento"] = "Evento desconhecido"

            elif problema == "id_animal_ausente":
                registro["id_animal"] = None

            registrar_problema(registro, problema)

    return len(selecionados)


# 12. GERAÇÃO DOS ANIMAIS E DOS CONJUNTOS RELACIONADOS

print("Iniciando geração dos dados sintéticos...")

animais = []
vacinacoes = []
diagnosticos = []
maus_tratos = []
obitos_eutanasias = []

# Recupera o próximo número da sequência.
proximo_numero = cursor.execute("""
SELECT proximo_valor
FROM controle_sequencias
WHERE nome = 'animais'
""").fetchone()[0]

for i in range(QUANTIDADE_ANIMAIS):

    numero = proximo_numero + i

    animal = gerar_animal(numero)

    animais.append(animal)

    vacinacoes.extend(
        gerar_vacinacoes(animal)
    )

    diagnosticos.extend(
        gerar_diagnosticos(animal)
    )

    maus_tratos.extend(
        gerar_maus_tratos(animal)
    )

    obitos_eutanasias.extend(
        gerar_eventos(animal)
    )

    if (i + 1) % 1000 == 0:
        print(f"{i + 1} animais gerados...")


# 13. PROBLEMAS DE STATUS APÓS EVENTO

# Introduz deliberadamente alguns casos de animal ativo
# mesmo possuindo um evento de óbito ou eutanásia.

animais_com_evento = {
    evento["_meta"]["id_animal_origem"]
    for evento in obitos_eutanasias
}

animais_por_origem = {
    animal["_meta"]["id_animal_origem"]: animal
    for animal in animais
}

if animais_com_evento:

    candidatos = [
        animais_por_origem[id_origem]
        for id_origem in animais_com_evento
        if id_origem in animais_por_origem
    ]

    quantidade_status_incorreto = math.ceil(
        len(candidatos) * 0.10
    )

    for animal in random.sample(
        candidatos,
        min(quantidade_status_incorreto, len(candidatos))
    ):
        animal["status"] = "Ativo"
        registrar_problema(
            animal,
            "animal_ativo_apos_obito_ou_eutanasia"
        )


# 14. APLICAÇÃO DOS PROBLEMAS POR CONJUNTO

relatorio_problemas = {}

relatorio_problemas["animais"] = aplicar_problemas(
    animais,
    "animais",
    TAXA_PROBLEMAS
)

relatorio_problemas["vacinacoes"] = aplicar_problemas(
    vacinacoes,
    "vacinacoes",
    TAXA_PROBLEMAS
)

relatorio_problemas["diagnosticos"] = aplicar_problemas(
    diagnosticos,
    "diagnosticos",
    TAXA_PROBLEMAS
)

relatorio_problemas["maus_tratos"] = aplicar_problemas(
    maus_tratos,
    "maus_tratos",
    TAXA_PROBLEMAS
)

relatorio_problemas["obitos_eutanasias"] = aplicar_problemas(
    obitos_eutanasias,
    "obitos_eutanasias",
    TAXA_PROBLEMAS
)


# 15. GERAÇÃO DE DUPLICIDADES

quantidade_duplicados = math.ceil(
    len(animais) * TAXA_DUPLICADOS
)

duplicados_exatos = []
duplicados_parciais = []

for animal in random.sample(
    animais,
    min(quantidade_duplicados, len(animais))
):

    copia = deepcopy(animal)

    if random.random() < 0.5:

        # Duplicidade exata: todos os campos são iguais.
        registrar_problema(
            copia,
            "duplicidade_exata"
        )

        duplicados_exatos.append(copia)

    else:

        # Duplicidade parcial: mesmo ID técnico e público,
        # mas com divergência em um campo.
        campo = random.choice([
            "nome",
            "raca",
            "peso_kg",
            "municipio",
            "sexo"
        ])

        if campo == "nome":
            copia["nome"] = "Nome divergente"

        elif campo == "raca":
            copia["raca"] = "Raça divergente"

        elif campo == "peso_kg":
            copia["peso_kg"] = round(
                random.uniform(1, 50), 2
            )

        elif campo == "municipio":
            copia["municipio"] = random.choice(municipios)

        elif campo == "sexo":
            copia["sexo"] = (
                "Fêmea" if animal["sexo"] == "Macho" else "Macho"
            )

        registrar_problema(
            copia,
            "duplicidade_parcial"
        )

        duplicados_parciais.append(copia)


animais.extend(duplicados_exatos)
animais.extend(duplicados_parciais)


# 16. ORGANIZAÇÃO DOS CONJUNTOS

conjuntos = {
    "animais": animais,
    "vacinacoes": vacinacoes,
    "diagnosticos": diagnosticos,
    "maus_tratos": maus_tratos,
    "obitos_eutanasias": obitos_eutanasias
}


# 17. PREPARAÇÃO DOS REGISTROS BRUTOS

registros_para_inserir = []

for nome_conjunto, registros in conjuntos.items():

    for registro in registros:

        registros_para_inserir.append((
            nome_conjunto,
            json.dumps(
                registro,
                ensure_ascii=False,
                allow_nan=False
            )
        ))


# 18. VERIFICAÇÃO DO MODO DE GERAÇÃO

quantidade_animais_anteriores = cursor.execute("""
SELECT COUNT(*)
FROM registros_brutos
WHERE conjunto = 'animais'
""").fetchone()[0]

if MODO_GERACAO == "INICIAL":

    if quantidade_animais_anteriores > 0:
        conexao.close()
        raise Exception(
            "Já existem animais brutos no banco. "
            "Altere MODO_GERACAO para 'ADICIONAR' "
            "se deseja gerar outro lote."
        )

elif MODO_GERACAO != "ADICIONAR":

    conexao.close()
    raise ValueError(
        "MODO_GERACAO deve ser 'INICIAL' ou 'ADICIONAR'."
    )


# 19. GRAVAÇÃO NO SQLITE

try:

    cursor.execute("BEGIN IMMEDIATE")

    # Atualiza a sequência de IDs.
    cursor.execute("""
    UPDATE controle_sequencias
    SET proximo_valor = ?
    WHERE nome = 'animais'
    """, (proximo_numero + QUANTIDADE_ANIMAIS,))

    # Cria o lote.
    colunas_lotes = {
        linha[1]
        for linha in cursor.execute(
            "PRAGMA table_info(lotes)"
        ).fetchall()
    }

    dados_lote = {
        "tipo_lote": "Geração sintética",
        "quantidade_informada": QUANTIDADE_ANIMAIS,
        "quantidade_gerada": len(animais),
        "data_criacao": datetime.now().isoformat(
            timespec="seconds"
        ),
        "descricao": (
            "Dados sintéticos de zoonoses. "
            "Inclui problemas controlados e duplicidades."
        ),
        "status": "Concluído"
    }

    campos_lote = [
        campo for campo in dados_lote
        if campo in colunas_lotes
    ]

    if not campos_lote:
        raise Exception(
            "Não foi possível identificar os campos da tabela lotes."
        )

    sql_lote = (
        f"INSERT INTO lotes "
        f"({', '.join(campos_lote)}) "
        f"VALUES ({', '.join(['?'] * len(campos_lote))})"
    )

    cursor.execute(
        sql_lote,
        [dados_lote[campo] for campo in campos_lote]
    )

    id_lote = cursor.lastrowid

    # Insere os registros brutos.
    cursor.executemany("""
    INSERT INTO registros_brutos
    (id_lote, conjunto, dados_json, data_recebimento)
    VALUES (?, ?, ?, ?)
    """, [
        (
            id_lote,
            conjunto,
            dados_json,
            datetime.now().isoformat(timespec="seconds")
        )
        for conjunto, dados_json in registros_para_inserir
    ])

    # Registra a operação na auditoria.
    colunas_auditoria = {
        linha[1]
        for linha in cursor.execute(
            "PRAGMA table_info(auditoria)"
        ).fetchall()
    }

    dados_auditoria = {
        "entidade": "lotes",
        "id_registro": str(id_lote),
        "operacao": "GERAÇÃO",
        "descricao": (
            f"Lote {id_lote}: "
            f"{len(animais)} animais e "
            f"{len(registros_para_inserir)} registros brutos."
        ),
        "data_operacao": datetime.now().isoformat(
            timespec="seconds"
        )
    }

    campos_auditoria = [
        campo for campo in dados_auditoria
        if campo in colunas_auditoria
    ]

    if campos_auditoria:

        sql_auditoria = (
            f"INSERT INTO auditoria "
            f"({', '.join(campos_auditoria)}) "
            f"VALUES ({', '.join(['?'] * len(campos_auditoria))})"
        )

        cursor.execute(
            sql_auditoria,
            [
                dados_auditoria[campo]
                for campo in campos_auditoria
            ]
        )

    conexao.commit()

except Exception:

    conexao.rollback()
    conexao.close()
    raise


# 20. RELATÓRIO DA GERAÇÃO

print("\n" + "=" * 60)
print("GERAÇÃO CONCLUÍDA")
print("=" * 60)

print(f"\nLote gerado: {id_lote}")
print(f"Modo: {MODO_GERACAO}")

print("\nQUANTIDADE DE REGISTROS POR CONJUNTO")

for nome, registros in conjuntos.items():
    print(f"{nome}: {len(registros)}")

print("\nPROBLEMAS INTRODUZIDOS NOS REGISTROS ORIGINAIS")

for nome, quantidade in relatorio_problemas.items():

    total = (
        len(conjuntos[nome])
        - (
            len(duplicados_exatos)
            + len(duplicados_parciais)
            if nome == "animais"
            else 0
        )
    )

    percentual = (
        quantidade / total * 100
        if total > 0 else 0
    )

    print(
        f"{nome}: {quantidade} registros "
        f"({percentual:.2f}%)"
    )

print("\nDUPLICIDADES DE ANIMAIS")

print(f"Duplicidades exatas: {len(duplicados_exatos)}")
print(f"Duplicidades parciais: {len(duplicados_parciais)}")
print(f"Total adicional: {quantidade_duplicados}")

print("\nTOTAL DE REGISTROS BRUTOS INSERIDOS")
print(len(registros_para_inserir))

print("\n" + "=" * 60)

conexao.close()

print("Banco SQLite atualizado com sucesso.")
print("Os dados anteriores foram preservados.")


In [ ]:

import sqlite3

# Reutiliza CAMINHO_BANCO definido na primeira celula.

conexao = sqlite3.connect(CAMINHO_BANCO)
cursor = conexao.cursor()

tabelas = [
    "registros_brutos",
    "lotes",
    "animais",
    "vacinacoes",
    "diagnosticos",
    "maus_tratos",
    "obitos_eutanasias",
    "auditoria"
]

for tabela in tabelas:
    quantidade = cursor.execute(
        f"SELECT COUNT(*) FROM {tabela}"
    ).fetchone()[0]

    print(f"{tabela}: {quantidade} registros")

conexao.close()
